# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nish-debug15/internship_flyrankai/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


- One row represents one pseudonymized client, one pseudonymized content item, and one report date from the `fact_content_daily_performance` table.
- The analysis uses the month `2026-03`, which is a mid-panel month.
- June 2026 (`_sample`) is intentionally excluded because it is the final month and reserved for testing, as recommended in the assignment.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN 'YOUR_HF_TOKEN'
)
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature / Label / Context / Excluded

| Category | Fields | Reason |
|----------|--------|--------|
| **Features** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions` | Historical metrics available before making a prediction. |
| **Label / Proxy** | Future content performance (not created in this notebook) | Represents the outcome that could be predicted in a future modeling task. |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` | Used to identify observations and support grouping or joins, not as predictive features. |
| **Excluded** | June 2026 data, future metrics | Excluded to prevent target leakage and preserve an honest evaluation. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [7]:
con.sql(f"""
SELECT *
FROM glob('{warehouse}/**')
LIMIT 20
""").df()

,file
0,hf://datasets/FlyRank/internship-warehouse/.gi...
1,hf://datasets/FlyRank/internship-warehouse/REA...
2,hf://datasets/FlyRank/internship-warehouse/dim...
3,hf://datasets/FlyRank/internship-warehouse/dim...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [10]:
con.sql(f"""
SELECT *
FROM read_parquet('{warehouse}/dim_clients.parquet')
LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [11]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [12]:
grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()

grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


### Verification Result — Grain

No duplicate rows were found for the combination of `report_date`, `client_hash_id`, and `content_hash_id`.

This confirms that one row represents one client, one content item, and one report date.

In [13]:
summary = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()

summary

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Verification Result — Row Count and Date Window

The March 2026 partition contains **9,841,378** rows.

The data spans the complete month, from **2026-03-01** through **2026-03-31**.

In [14]:
availability = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Verification Result — Data Availability

Filtering with `gsc_data_available IS TRUE` leaves **3,611,061** rows.

This confirms that Google Search Console data is available for only a subset of the observations.

In [15]:
features = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10
""").df()

features

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


## Feature Availability

| Feature | Knowable at the decision moment because... |
|----------|--------------------------------------------|
| `gsc_impressions` | It is historical search impression data available before making a prediction. |
| `gsc_clicks` | It is historical click data collected before the prediction time. |
| `gsc_avg_position` | It is calculated from historical search rankings and is known beforehand. |
| `ga4_pageviews` | It is historical website traffic data already recorded before prediction. |
| `ga4_sessions` | It is historical session information available before making a prediction. |

## 4. Data Limits

This dataset is an unbalanced panel, meaning different clients have different amounts of historical data available. Some observations contain only Google Search Console or only Google Analytics 4 data depending on when tracking began. Therefore, the results should be interpreted as observational and decision-support rather than directly comparable across every client.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.